# 03 — Exp 1: Main Timing Benchmark (`When2Speak`)

1. Calibrate $(\lambda, \tau_1)$ strictly on `When2Speak-ValCalib` ($\le 200$ validation turns — **Invariant #8**) independently for `Luna` and `Terra`.
2. Evaluate `B0 Direct Chat`, `B1 Single Gate` (LLM & Jev), `B2 ProACT-Reimpl`, and `DIAL-Skill` (`Luna` & `Terra`).
3. Compute McNemar's test + Bootstrap 95% CIs and persist `metrics_summary.json` to Drive.
4. Default `MODE = 'pilot'` (400 turns); switch to `MODE = 'main'` (2,000 turns) for paper Table 1.

In [7]:
# === CELL 0: Mount Google Drive & Configure Persistent Workspace ===
import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/when2speak_research")
else:
    DRIVE_ROOT = Path("./local_drive_mirror").resolve()

for sub in [
    "secrets",
    "config_cache",
    "datasets/pagemantle",
    "datasets/when2speak",
    "datasets/pagemantle_ext",
    "caches",
    "runs",
    "paper_artifacts/tables",
    "paper_artifacts/figures",
]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["DIAL_SKILL_DRIVE_ROOT"] = str(DRIVE_ROOT)
print("Persistent DRIVE_ROOT:", DRIVE_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistent DRIVE_ROOT: /content/drive/MyDrive/when2speak_research


In [8]:
# === CELL 1: Sync GitHub Repo (git clone / git pull) & Hot-Reload ===
REPO_URL = "https://github.com/BoldHu/multiagent-speak.git"  # Edit once with your repo URL
REPO_DIR = Path("/content/when2speak") if IN_COLAB else Path(".").resolve()

import importlib, sys, types

if IN_COLAB:
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}
    try:
        import typesafe_sdk, duckdb, typer, tenacity  # noqa: F401
        !pip install -q -e . --no-deps
    except ImportError:
        !pip install -q -e .

# Ensure `src/` is immediately importable in a fresh Colab kernel without restarting Python
_src_dir = str((REPO_DIR / "src").resolve())
if _src_dir not in sys.path:
    sys.path.insert(0, _src_dir)

# Colab Python 3.12+/3.13 removed stdlib `imp`, which older IPython autoreload still imports.
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp_shim = types.ModuleType("imp")
        _imp_shim.reload = importlib.reload
        sys.modules["imp"] = _imp_shim

try:
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("load_ext", "autoreload")
        _ip.run_line_magic("autoreload", "2")
except Exception as _exc:
    print(f"Skipping autoreload ({_exc}); restart runtime after git pull if needed.")


Already up to date.
/content/when2speak
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for dial-skill (pyproject.toml) ... done
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
# === CELL 2: Bootstrap Secrets & Resolved Model Config (Never Prints Raw Keys) ===
import json
from dial_skill.utils.secrets import describe_secret_sources
from dial_skill.llm.preflight import load_resolved_models, resolve_model_for_role
from dial_skill.config import load_models_config, load_dial_skill_config, load_experiments_config

print("Secret lookup diagnostics:", json.dumps(describe_secret_sources(drive_root=DRIVE_ROOT), indent=2))
MODELS_CFG = load_models_config()
DIAL_CFG = load_dial_skill_config()
EXP_CFG = load_experiments_config()
RESOLVED_MODELS = load_resolved_models(drive_root=DRIVE_ROOT)
print("Resolved models:", RESOLVED_MODELS.model_dump() if RESOLVED_MODELS else "Not yet generated (run Notebook 00)")


Secret lookup diagnostics: {
  "jev": {
    "status": "found",
    "source": "drive:JEV_API_KEY.txt",
    "masked": "apik...c3aa"
  },
  "openai": {
    "status": "found",
    "source": "drive:OPENAI_KEY.txt",
    "masked": "sk-p...yN8A"
  }
}
Resolved models: {'timestamp_utc': '2026-11-20T13:25:11.705978+00:00', 'luna': 'gpt-4.1-mini', 'terra': 'gpt-4.1', 'sol': 'gpt-6.1-sol', 'jev_model': 'jev-latest', 'jev_healthy': True, 'triage_backend': 'jev', 'available_openai_models_count': 99, 'notes': ["Probe failed for luna candidate 'gpt-6-luna': BadRequestError", "Probe failed for luna candidate 'gpt-5.6-luna': BadRequestError", "Probe failed for terra candidate 'gpt-6.1-terra': BadRequestError", "Probe failed for terra candidate 'gpt-5.6-terra': BadRequestError"]}


## Step 1: Select Mode & Print Cost Estimate

In [10]:
from dial_skill.eval.runners import estimate_run_cost_usd
from dial_skill.utils.checkpoint import save_run_metadata

MODE = "main"  # Change to "main" (2,000 turns) for full paper Table 1!
RUN_DIR = DRIVE_ROOT / "runs" / "main_when2speak" / MODE
est = estimate_run_cost_usd(mode=MODE, include_pagemantle=False)
print("Estimated Cost:", json.dumps(est, indent=2))
save_run_metadata(RUN_DIR, {"mode": MODE, "estimate": est})


Estimated Cost: {
  "mode": "main",
  "estimated_api_calls": 4400,
  "estimated_when2speak_usd": 4.4,
  "estimated_pagemantle_usd": 0.0,
  "estimated_ablations_usd": 0.0,
  "estimated_total_usd": 4.4
}


PosixPath('/content/drive/MyDrive/when2speak_research/runs/main_when2speak/main/run_config.json')

## Step 2: Validation Calibration (`val_calib <= 200` turns, Zero Leakage — Independent for Luna & Terra)

In [11]:
from dial_skill.benchmarks.when2speak import assert_no_calibration_leakage, load_when2speak_split
from dial_skill.core.pipeline import DialSkillAgent
from dial_skill.core.skill_store import SkillStore
from dial_skill.eval.runners import calibrate_when2speak_thresholds, run_when2speak_eval
from dial_skill.llm.jev_client import TriageClient
from dial_skill.llm.openai_client import OpenAIClient
from dial_skill.utils.cache import SQLiteAPICache

cache = SQLiteAPICache(DRIVE_ROOT / "caches" / "api_cache.sqlite")
oai = OpenAIClient(models_cfg=MODELS_CFG, cache=cache)
luna_model = resolve_model_for_role("luna", MODELS_CFG, RESOLVED_MODELS)
terra_model = resolve_model_for_role("terra", MODELS_CFG, RESOLVED_MODELS)
backend = RESOLVED_MODELS.triage_backend if RESOLVED_MODELS else MODELS_CFG.roles.triage_backend
triage_client = TriageClient(models_cfg=MODELS_CFG, cache=cache, openai_client=oai, luna_model=luna_model, backend=backend)

val_turns = load_when2speak_split("val_calib", local_dir=DRIVE_ROOT / "datasets" / "when2speak", seed=EXP_CFG["seed"])
eval_turns = load_when2speak_split(MODE, local_dir=DRIVE_ROOT / "datasets" / "when2speak", seed=EXP_CFG["seed"])
assert_no_calibration_leakage(val_turns, eval_turns, max_calib_turns=200)

# Calibrate Luna (gpt-4.1-mini) strictly on validation split <= 200 turns
calib_luna_agent = DialSkillAgent(
    triage_client=triage_client, openai_client=oai, gate_model=luna_model, executor_model=luna_model, cfg=DIAL_CFG
)
val_eps_luna = await run_when2speak_eval(
    agent=calib_luna_agent, turns=val_turns, output_jsonl=RUN_DIR / "val_calib_dial.jsonl", resume=True
)
calib_luna = calibrate_when2speak_thresholds(
    val_turns=val_turns,
    val_episodes=val_eps_luna,
    eval_turns=eval_turns,
    lambda_grid=EXP_CFG["when2speak"]["calibration"]["lambda_grid"],
    tau_dm_grid=EXP_CFG["when2speak"]["calibration"]["tau_dm_grid"],
    theta_floor=DIAL_CFG.gate.theta_floor,
    max_fir=0.11,
    max_mir=0.45,
)

# Calibrate Terra (gpt-4.1) independently on the exact same validation split <= 200 turns
calib_terra_agent = DialSkillAgent(
    triage_client=triage_client, openai_client=oai, gate_model=terra_model, executor_model=terra_model, cfg=DIAL_CFG
)
val_eps_terra = await run_when2speak_eval(
    agent=calib_terra_agent, turns=val_turns, output_jsonl=RUN_DIR / "val_calib_dial_terra.jsonl", resume=True
)
calib_terra = calibrate_when2speak_thresholds(
    val_turns=val_turns,
    val_episodes=val_eps_terra,
    eval_turns=eval_turns,
    lambda_grid=EXP_CFG["when2speak"]["calibration"]["lambda_grid"],
    tau_dm_grid=EXP_CFG["when2speak"]["calibration"]["tau_dm_grid"],
    theta_floor=DIAL_CFG.gate.theta_floor,
    max_fir=0.11,
    max_mir=0.45,
)

calib = calib_luna
calib_bundle = {"luna": calib_luna, "terra": calib_terra, **calib_luna}
print("Calibrated scalars (Luna):", json.dumps(calib_luna, indent=2))
print("Calibrated scalars (Terra):", json.dumps(calib_terra, indent=2))
(RUN_DIR / "calibration.json").write_text(json.dumps(calib_bundle, indent=2), encoding="utf-8")


W2S[val_calib_dial]:   0%|          | 0/200 [00:00<?, ?it/s]

W2S[val_calib_dial_terra]:   0%|          | 0/200 [00:00<?, ?it/s]

Calibrated scalars (Luna): {
  "best_lambda": 0.5,
  "best_tau_dm": 0.7,
  "val_macro_f1": 0.62018,
  "val_fir": 0.044444,
  "val_mir": 0.75,
  "n_val_turns": 200
}
Calibrated scalars (Terra): {
  "best_lambda": 0.75,
  "best_tau_dm": 0.7,
  "val_macro_f1": 0.596118,
  "val_fir": 0.094444,
  "val_mir": 0.7,
  "n_val_turns": 200
}


465

## Step 3: Run Baselines (`B0`, `B1`, `B2 ProACT`) & `DIAL-Skill` (`Luna` & `Terra`)

In [12]:
from dial_skill.baselines.direct_chat import DirectChatBaseline
from dial_skill.baselines.proact import ProACTBaseline
from dial_skill.baselines.single_gate import SingleGateBaseline
from dial_skill.eval.metrics import compute_when2speak_metrics, mcnemar_test

tuned_cfg_luna = load_dial_skill_config(
    overrides={"gate": {"lambda_disruption": calib_luna["best_lambda"], "tau_dm": calib_luna["best_tau_dm"]}}
)
tuned_cfg_terra = load_dial_skill_config(
    overrides={"gate": {"lambda_disruption": calib_terra["best_lambda"], "tau_dm": calib_terra["best_tau_dm"]}}
)

methods = {
    "B0_direct_luna": DirectChatBaseline(openai_client=oai, model=luna_model),
    "B1_single_llm_luna": SingleGateBaseline(openai_client=oai, gate_model=luna_model, gate_source="llm"),
    "B1_single_jev": SingleGateBaseline(openai_client=oai, gate_model=luna_model, triage_client=triage_client, gate_source="jev"),
    "B2_proact_luna": ProACTBaseline(openai_client=oai, gate_model=luna_model),
    "DIAL_Skill_luna": DialSkillAgent(
        triage_client=triage_client, openai_client=oai, gate_model=luna_model, executor_model=luna_model, cfg=tuned_cfg_luna, method_name="DIAL_Skill_luna"
    ),
    "DIAL_Skill_terra": DialSkillAgent(
        triage_client=triage_client, openai_client=oai, gate_model=terra_model, executor_model=terra_model, cfg=tuned_cfg_terra, method_name="DIAL_Skill_terra"
    ),
}

y_true = [t.label_speak for t in eval_turns]
results_summary = {}
preds_by_method = {}

for name, agent in methods.items():
    eps = await run_when2speak_eval(
        agent=agent, turns=eval_turns, output_jsonl=RUN_DIR / f"preds_{name}.jsonl", resume=True, generate_text=False
    )
    preds_by_method[name] = [e.gate.binary_speak for e in eps]
    results_summary[name] = compute_when2speak_metrics(y_true, eps)
    m = results_summary[name]
    print(f"[{name:20s}] Macro F1={m['macro_f1']:.3f} | Speak F1={m['speak_f1']:.3f} | FIR={m['fir']:.3f} | MIR={m['mir']:.3f} | AUROC={m['auroc']:.3f} | TokRed={m['token_reduction_pct']:.1f}%")

sig_vs_direct = mcnemar_test(y_true, preds_by_method["DIAL_Skill_luna"], preds_by_method["B0_direct_luna"])
sig_vs_proact = mcnemar_test(y_true, preds_by_method["DIAL_Skill_luna"], preds_by_method["B2_proact_luna"])
sig_vs_b1_jev = mcnemar_test(y_true, preds_by_method["DIAL_Skill_luna"], preds_by_method["B1_single_jev"])
sig_terra_vs_direct = mcnemar_test(y_true, preds_by_method["DIAL_Skill_terra"], preds_by_method["B0_direct_luna"])

print("McNemar DIAL_Luna vs Direct:", sig_vs_direct)
print("McNemar DIAL_Luna vs ProACT:", sig_vs_proact)
print("McNemar DIAL_Luna vs B1_Jev:", sig_vs_b1_jev)
print(f"Total run spend: OpenAI=${oai.usage.total_cost_usd:.4f}, Jev=${triage_client.total_cost_usd:.4f}")

summary_payload = {
    "mode": MODE,
    "calibration": calib_bundle,
    "metrics": results_summary,
    "mcnemar_dial_vs_direct": sig_vs_direct,
    "mcnemar_dial_vs_proact": sig_vs_proact,
    "mcnemar_dial_vs_b1_jev": sig_vs_b1_jev,
    "mcnemar_terra_vs_direct": sig_terra_vs_direct,
    "triage_meta": triage_client.get_run_metadata(),
}
(RUN_DIR / "metrics_summary.json").write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")
print("Saved metrics_summary.json to Drive.")


W2S[preds_B0_direct_luna]:   0%|          | 0/2000 [00:00<?, ?it/s]

[B0_direct_luna      ] Macro F1=0.400 | Speak F1=0.243 | FIR=0.598 | MIR=0.281 | AUROC=0.560 | TokRed=0.0%


W2S[preds_B1_single_llm_luna]:   0%|          | 0/2000 [00:00<?, ?it/s]

[B1_single_llm_luna  ] Macro F1=0.454 | Speak F1=0.262 | FIR=0.502 | MIR=0.317 | AUROC=0.619 | TokRed=0.0%


W2S[preds_B1_single_jev]:   0%|          | 0/2000 [00:00<?, ?it/s]

[B1_single_jev       ] Macro F1=0.580 | Speak F1=0.287 | FIR=0.154 | MIR=0.651 | AUROC=0.649 | TokRed=0.0%


... (2 outputs left out)

W2S[preds_DIAL_Skill_luna]:   0%|          | 0/2000 [00:00<?, ?it/s]

[DIAL_Skill_luna     ] Macro F1=0.622 | Speak F1=0.318 | FIR=0.045 | MIR=0.751 | AUROC=0.689 | TokRed=41.6%


W2S[preds_DIAL_Skill_terra]:   0%|          | 0/2000 [00:00<?, ?it/s]

[DIAL_Skill_terra    ] Macro F1=0.656 | Speak F1=0.394 | FIR=0.079 | MIR=0.618 | AUROC=0.721 | TokRed=41.6%
McNemar DIAL_Luna vs Direct: {'n_a_correct_b_wrong': 978, 'n_b_correct_a_wrong': 127, 'chi2_stat': 653.846154, 'p_value': 0.0}
McNemar DIAL_Luna vs ProACT: {'n_a_correct_b_wrong': 374, 'n_b_correct_a_wrong': 115, 'chi2_stat': 136.122699, 'p_value': 0.0}
McNemar DIAL_Luna vs B1_Jev: {'n_a_correct_b_wrong': 243, 'n_b_correct_a_wrong': 77, 'chi2_stat': 85.078125, 'p_value': 0.0}
Total run spend: OpenAI=$13.3363, Jev=$3.4185
Saved metrics_summary.json to Drive.


In [13]:
models = sorted(m.id for m in (await oai._get_client().models.list()).data)
print([m for m in models if any(k in m for k in ("gpt-4o", "gpt-4.1", "gpt-5", "o3", "o4"))])

['gpt-4.1', 'gpt-4.1-2025-04-14', 'gpt-4.1-mini', 'gpt-4.1-mini-2025-04-14', 'gpt-4o', 'gpt-4o-2024-08-06', 'gpt-4o-2024-11-20', 'gpt-4o-mini', 'gpt-4o-mini-2024-07-18', 'gpt-4o-mini-transcribe', 'gpt-4o-mini-transcribe-2025-03-20', 'gpt-4o-mini-transcribe-2025-12-15', 'gpt-4o-mini-tts', 'gpt-4o-mini-tts-2025-03-20', 'gpt-4o-mini-tts-2025-12-15', 'gpt-4o-transcribe', 'gpt-4o-transcribe-diarize', 'gpt-5', 'gpt-5-2025-08-07', 'gpt-5-mini', 'gpt-5-mini-2025-08-07', 'gpt-5-nano', 'gpt-5-nano-2025-08-07', 'gpt-5-pro', 'gpt-5-pro-2025-10-06', 'gpt-5-search-api', 'gpt-5-search-api-2025-10-14', 'gpt-5.1', 'gpt-5.1-2025-11-13', 'gpt-5.2', 'gpt-5.2-2025-12-11', 'gpt-5.2-pro', 'gpt-5.2-pro-2025-12-11', 'gpt-5.3-codex', 'gpt-5.4', 'gpt-5.4-2026-03-05', 'gpt-5.4-mini', 'gpt-5.4-mini-2026-03-17', 'gpt-5.4-nano', 'gpt-5.4-nano-2026-03-17', 'gpt-5.4-pro', 'gpt-5.4-pro-2026-03-05', 'gpt-5.5', 'gpt-5.5-2026-04-23', 'gpt-5.5-pro', 'gpt-5.5-pro-2026-04-23', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 